In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn

from sklearn.metrics import mean_squared_error, mean_absolute_error

from scipy import stats

import xgboost as xgb

**Building sliding windows per enginer**

In [3]:
DATA_DIR = Path.cwd().parent / 'data' / 'processed'
train_split = pd.read_csv(DATA_DIR / 'train_split.csv')
val_split = pd.read_csv(DATA_DIR / 'val_split.csv')

feature_cols = [c for c in train_split.columns if 'sensor' in c]
WINDOW_SIZE = 30

def create_sequences(df, feature_cols, window_size, target_col='RUL'):
    X, y, units_used = [], [], []
    for unit in df['unit'].unique():
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        features = unit_data[feature_cols].values
        targets = unit_data[target_col].values
        n_cycles = len(unit_data)

        if n_cycles < window_size:
            # pad short-lived engines at the start by repeating the first row
            pad_len = window_size - n_cycles
            padding = np.repeat(features[0:1], pad_len, axis=0)
            features = np.vstack([padding, features])
            targets = np.concatenate([np.repeat(targets[0], pad_len), targets])
            n_cycles = window_size

        for i in range(n_cycles - window_size + 1):
            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])   # RUL at the END of the window
            units_used.append(unit)

    return np.array(X), np.array(y), np.array(units_used)

X_train, y_train, train_units_seq = create_sequences(train_split, feature_cols, WINDOW_SIZE)
X_val, y_val, val_units_seq = create_sequences(val_split, feature_cols, WINDOW_SIZE)

print("X_train shape:", X_train.shape)   # (num_sequences, 30, num_features)
print("X_val shape:", X_val.shape)

X_train shape: (14241, 30, 49)
X_val shape: (3490, 30, 49)


**Validation split**

In [4]:
train_units = train_split['unit'].unique()
fit_units, earlystop_units = train_test_split(train_units, test_size=0.15, random_state=42)

fit_df = train_split[train_split['unit'].isin(fit_units)]
earlystop_df = train_split[train_split['unit'].isin(earlystop_units)]

X_fit, y_fit, _ = create_sequences(fit_df, feature_cols, WINDOW_SIZE)
X_earlystop, y_earlystop, _ = create_sequences(earlystop_df, feature_cols, WINDOW_SIZE)

print("Fit:", X_fit.shape, "Early-stop monitor:", X_earlystop.shape)

Fit: (12041, 30, 49) Early-stop monitor: (2200, 30, 49)


**PyTorch Dataset and DataLoader**

In [5]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64
fit_loader = DataLoader(RULDataset(X_fit, y_fit), batch_size=BATCH_SIZE, shuffle=True)
earlystop_loader = DataLoader(RULDataset(X_earlystop, y_earlystop), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(RULDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

**LTSM Model**

In [6]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]   # take the output at the FINAL time step
        out = self.fc(last_hidden)
        return out.squeeze()

model = LSTMRegressor(input_size=len(feature_cols), hidden_size=64, num_layers=2)

**Training loop**

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
PATIENCE = 10
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    for X_batch, y_batch in fit_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    earlystop_losses = []
    with torch.no_grad():
        for X_batch, y_batch in earlystop_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            earlystop_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    earlystop_loss = np.mean(earlystop_losses)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.2f}, earlystop_loss={earlystop_loss:.2f}")

    if earlystop_loss < best_val_loss:
        best_val_loss = earlystop_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), '../models/lstm_best.pt')  # save best checkpoint
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

Epoch 1: train_loss=11172.60, earlystop_loss=9667.63
Epoch 2: train_loss=9096.64, earlystop_loss=8024.44
Epoch 3: train_loss=7616.73, earlystop_loss=6758.01
Epoch 4: train_loss=6517.46, earlystop_loss=5804.41
Epoch 5: train_loss=5679.05, earlystop_loss=5105.43
Epoch 6: train_loss=5084.99, earlystop_loss=4612.03
Epoch 7: train_loss=4718.36, earlystop_loss=4276.10
Epoch 8: train_loss=4398.42, earlystop_loss=4051.79
Epoch 9: train_loss=4210.98, earlystop_loss=3914.81
Epoch 10: train_loss=4113.25, earlystop_loss=3834.47
Epoch 11: train_loss=4044.58, earlystop_loss=3789.98
Epoch 12: train_loss=4057.38, earlystop_loss=3767.99
Epoch 13: train_loss=4013.77, earlystop_loss=3757.53
Epoch 14: train_loss=3983.86, earlystop_loss=3752.93
Epoch 15: train_loss=3978.25, earlystop_loss=3751.53
Epoch 16: train_loss=3975.43, earlystop_loss=3751.01
Epoch 17: train_loss=3980.39, earlystop_loss=3750.99
Epoch 18: train_loss=3973.64, earlystop_loss=3751.02
Epoch 19: train_loss=3986.56, earlystop_loss=3751.06
E

**Evaluate**

In [8]:
model.load_state_dict(torch.load('../models/lstm_best.pt'))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
mae = mean_absolute_error(all_targets, all_preds)
print(f"LSTM RMSE: {rmse:.2f} cycles")
print(f"LSTM MAE: {mae:.2f} cycles")

LSTM RMSE: 58.30 cycles
LSTM MAE: 48.89 cycles


**Save final model**

In [9]:
torch.save({
    'model_state_dict': model.state_dict(),
    'input_size': len(feature_cols),
    'hidden_size': 64,
    'num_layers': 2,
    'window_size': WINDOW_SIZE
}, '../models/lstm_final.pt')

**Align predictions**

In [10]:
# Rebuild val sequences, but track which (unit, cycle) each one ends on
def create_sequences_with_ids(df, feature_cols, window_size, target_col='RUL'):
    X, y, unit_ids, cycle_ids = [], [], [], []
    for unit in df['unit'].unique():
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        features = unit_data[feature_cols].values
        targets = unit_data[target_col].values
        cycles = unit_data['cycle'].values
        n_cycles = len(unit_data)

        if n_cycles < window_size:
            continue  # skip (FD001 units are all longer than 30 cycles, so this shouldn't trigger)

        for i in range(n_cycles - window_size + 1):
            X.append(features[i:i + window_size])
            y.append(targets[i + window_size - 1])
            unit_ids.append(unit)
            cycle_ids.append(cycles[i + window_size - 1])  # the cycle this prediction is FOR

    return np.array(X), np.array(y), np.array(unit_ids), np.array(cycle_ids)

In [11]:
X_val_seq, y_val_seq, val_unit_ids, val_cycle_ids = create_sequences_with_ids(
    val_split, feature_cols, WINDOW_SIZE
)

In [12]:
# Get LSTM predictions (same as before, but now we track which rows they belong to)
model.load_state_dict(torch.load('../models/lstm_best.pt'))
model.eval()

lstm_preds = []
with torch.no_grad():
    X_tensor = torch.tensor(X_val_seq, dtype=torch.float32).to(device)
    for i in range(0, len(X_tensor), 64):
        batch = X_tensor[i:i+64]
        preds = model(batch).cpu().numpy()
        lstm_preds.extend(preds)
lstm_preds = np.array(lstm_preds)

In [13]:
# Now pull the MATCHING XGBoost predictions for these exact (unit, cycle) pairs
comparison_df = pd.DataFrame({
    'unit': val_unit_ids,
    'cycle': val_cycle_ids,
    'true_RUL': y_val_seq,
    'lstm_pred': lstm_preds
})

In [14]:
xgb_val_preds = pd.read_csv('../data/processed/xgb_val_predictions.csv')

comparison_df = comparison_df.merge(xgb_val_preds, on=['unit', 'cycle'], how='inner')
print(comparison_df.shape)
comparison_df.head()

(3490, 5)


,unit,cycle,true_RUL,lstm_pred,xgb_pred
0,1,30,162,94.231384,198.08460
1,1,31,161,94.231384,185.31477
2,1,32,160,94.231384,191.56390
3,1,33,159,94.231384,195.17723
4,1,34,158,94.231384,183.80426


In [16]:
print(comparison_df.shape)
comparison_df.head()

(3490, 5)


,unit,cycle,true_RUL,lstm_pred,xgb_pred
0,1,30,162,94.231384,198.08460
1,1,31,161,94.231384,185.31477
2,1,32,160,94.231384,191.56390
3,1,33,159,94.231384,195.17723
4,1,34,158,94.231384,183.80426


**Computing per-sample errors for both models**

In [17]:
comparison_df['lstm_error'] = np.abs(comparison_df['true_RUL'] - comparison_df['lstm_pred'])
comparison_df['xgb_error'] = np.abs(comparison_df['true_RUL'] - comparison_df['xgb_pred'])

print("XGBoost MAE:", comparison_df['xgb_error'].mean())
print("LSTM MAE:", comparison_df['lstm_error'].mean())

XGBoost MAE: 22.182440307842406
LSTM MAE: 48.88846106761506


**The actual hypothesis test**

In [18]:
errors_diff = comparison_df['xgb_error'] - comparison_df['lstm_error']

# Paired t-test (parametric)
t_stat, p_value_ttest = stats.ttest_rel(comparison_df['xgb_error'], comparison_df['lstm_error'])

# Wilcoxon signed-rank test (non-parametric, more robust to outliers)
w_stat, p_value_wilcoxon = stats.wilcoxon(comparison_df['xgb_error'], comparison_df['lstm_error'])

print(f"Paired t-test: t={t_stat:.3f}, p={p_value_ttest:.4f}")
print(f"Wilcoxon signed-rank: W={w_stat:.1f}, p={p_value_wilcoxon:.4f}")
print(f"\nMean error difference (XGBoost - LSTM): {errors_diff.mean():.3f}")
print(f"  (positive = LSTM has lower error on average)")

Paired t-test: t=-39.115, p=0.0000
Wilcoxon signed-rank: W=1108259.0, p=0.0000

Mean error difference (XGBoost - LSTM): -26.706
  (positive = LSTM has lower error on average)


**Adding Bootstrap confidence interval**

In [19]:
np.random.seed(42)
n_bootstrap = 10000
n_samples = len(comparison_df)
bootstrap_diffs = []

for _ in range(n_bootstrap):
    sample_idx = np.random.choice(n_samples, size=n_samples, replace=True)
    sample = errors_diff.values[sample_idx]
    bootstrap_diffs.append(sample.mean())

ci_lower, ci_upper = np.percentile(bootstrap_diffs, [2.5, 97.5])
print(f"95% CI for mean error difference (XGBoost - LSTM): [{ci_lower:.3f}, {ci_upper:.3f}]")

95% CI for mean error difference (XGBoost - LSTM): [-28.034, -25.341]


**Model Comparison: XGBoost vs. LSTM**

Both models were evaluated on the identical set of held-out validation engines and cycles (3,490 samples), using mean absolute error (MAE) in cycles.

| Model   | MAE (cycles) |
|---------|--------------|
| XGBoost | 22.18        |
| LSTM    | 48.89        |

A paired Wilcoxon signed-rank test on per-sample absolute errors gave p < 0.0001, and a bootstrap 95% CI on the mean error difference (XGBoost − LSTM) was [-28.03, -25.34].

**Result**: The difference was highly statistically significant. XGBoost outperformed the LSTM by an average of ~26.7 cycles per prediction, roughly halving the error.

**Interpretation**: Despite the LSTM's ability to model raw temporal sequences directly, it did not outperform XGBoost with hand-engineered rolling statistics; in fact it performed substantially worse. This suggests the rolling mean/std features already captured most of the temporal signal relevant to this task, and FD001's single operating condition/single fault mode likely doesn't contain enough complex sequential structure to reward a more expressive sequence model. It's also possible the LSTM is undertrained relative to its capacity: with only ~12,000 training sequences and a 2-layer, modest hidden size, there may be room to improve it further (longer training, tuning window size, or additional regularization) before drawing a final conclusion about which architecture is better suited to this problem versus which one happened to be better tuned here.

**Caveat**: this comparison is based on a single train/validation split; a more rigorous version would repeat this across multiple random unit-level splits to check whether the ~27-cycle gap is stable, rather than an artifact of this particular split.